# FOCS: SSC check model on Kaggle's free GPU

Serves **Qwen3-8B (AWQ)** with vLLM on the 2×T4 and opens a temporary public link to it (Cloudflare quick tunnel), so `import_ssc.py --remote` on the laptop can check the SSC questions many at a time.

**Before running:** Settings (right panel) → Accelerator **GPU T4 x2**, Internet **On** (needs a phone-verified account).
Run all cells, then copy the `URL` and `KEY` printed by the last cell. Keep this tab open: the session lasts up to 12 hours.

In [ ]:
!pip install -q "vllm==0.10.2" "transformers>=4.56,<5" 2>&1 | tail -2
!wget -q https://github.com/cloudflare/cloudflared/releases/download/2025.8.1/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, secrets, subprocess, time, re, urllib.request
KEY = secrets.token_hex(16)
# T4 (Turing): no bfloat16, so float16. Two things made it crawl at 3 tokens/s:
# - vLLM's default attention on Turing is a slow fallback: force the Triton kernel;
# - one model split over both T4s (tensor parallel) talks over PCIe on every token: instead a whole copy on each
#   GPU (data parallel; 8B AWQ fits in one T4), so the two work side by side.
env = {**os.environ, "VLLM_ATTENTION_BACKEND": "TRITON_ATTN_VLLM_V1"}
server = subprocess.Popen(
    ["vllm", "serve", "Qwen/Qwen3-8B-AWQ", "--dtype", "float16", "--data-parallel-size", "2",
     "--max-model-len", "8192", "--gpu-memory-utilization", "0.90", "--max-num-seqs", "32",
     "--reasoning-parser", "qwen3", "--api-key", KEY, "--port", "8000"],
    stdout=open("/kaggle/working/vllm.log", "w"), stderr=subprocess.STDOUT, env=env,
    start_new_session=True)  # its own process group: stopping a cell (Cancel Run) no longer kills it
for _ in range(180):  # model download + load: a few minutes
    try:
        urllib.request.urlopen("http://localhost:8000/health"); break
    except Exception:
        if server.poll() is not None:
            raise SystemExit(open("/kaggle/working/vllm.log").read()[-3000:])
        time.sleep(10)
print("vLLM is up")

In [ ]:
# Speed test: 32 questions at once. A T4 pair should give several hundred tokens/s; under ~100, tell Claude.
import json, concurrent.futures
def ask(_):
    req = urllib.request.Request("http://localhost:8000/v1/chat/completions", method="POST",
        headers={"Content-Type": "application/json", "Authorization": f"Bearer {KEY}"},
        data=json.dumps({"model": "Qwen/Qwen3-8B-AWQ", "max_tokens": 300, "temperature": 0,
                         "messages": [{"role": "user", "content": "Explain step by step: 17% of 640 + 23% of 360"}],
                         "chat_template_kwargs": {"enable_thinking": False}}).encode())
    return json.load(urllib.request.urlopen(req))["usage"]["completion_tokens"]
t = time.time()
with concurrent.futures.ThreadPoolExecutor(32) as pool:
    tokens = sum(pool.map(ask, range(32)))
print(f"{tokens} tokens in {time.time() - t:.0f} s = {tokens / (time.time() - t):.0f} tokens/s")

In [ ]:
# The tunnel writes to a file (nothing to drain) in its own process group, so stopping a cell doesn't kill it
tunnel = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", "http://localhost:8000"],
                          stdout=open("/kaggle/working/tunnel.log", "w"), stderr=subprocess.STDOUT,
                          start_new_session=True)
URL = None
while not URL:
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/kaggle/working/tunnel.log").read())
    URL = m and m.group(0)
print("URL =", URL)
print("KEY =", KEY)

In [ ]:
# Keeps the session busy (Kaggle stops an idle one) and shows the server's progress every 5 minutes.
# Stopping this cell doesn't stop the server or the tunnel; run it again any time.
import time
while True:
    time.sleep(300)
    lines = open("/kaggle/working/vllm.log").read().strip().splitlines()
    print(time.strftime("%H:%M"), lines[-1][:200] if lines else "")